# Iris classification — Notebook
This notebook demonstrates reading the included `jupyter/data/iris.csv`, exporting the small `iris.fuse` model to ONNX, and running a quick inference with `onnxruntime`.

In [ ]:
# 1) Inspect the dataset
import pandas as pd
pd.options.display.width = 120
df = pd.read_csv('../data/iris.csv')
df.head()

The model is a tiny centroid-based classifier (`jupyter/cookbook/iris.fuse`) that picks the nearest of three centroids and returns the class index (0:setosa, 1:versicolor, 2:virginica).

In [ ]:
# 2) Export to ONNX using the local fuse binary (adjust path if your venv is different)
!./.venv/bin/fuse onnx -f iris.fuse -o onnx/cookbook/iris.onnx

In [ ]:
# 3) Load the ONNX model and run inference on a sample row
import onnx
import onnxruntime as ort
import numpy as np

model_path = 'onnx/cookbook/iris.onnx'
onnx_model = onnx.load(model_path)
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession(model_path)
# pick the first sample from the CSV
x = df.iloc[0, :4].to_numpy(dtype=np.float32)
# `iris.fuse` expects a shape (4,) vector named 'x'
pred = sess.run(None, {sess.get_inputs()[0].name: x})
print('raw prediction output:', pred)

You're all set — the `@proof` checks embedded in `iris.fuse` can be validated by running the cookbook test harness (e.g., `python -m pytest tests/test_cookbook.py -q -k iris` when a test is present).